# Network Momentum — graph and full backtest validation

This notebook inspects the completed 2000–2022 fixed-hyperparameter approximation. It loads saved artifacts and does not recompute the expensive daily graph history.

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
OUTPUTS = PROJECT_ROOT / 'results' / 'replication' / 'artifacts'
print('Project root:', PROJECT_ROOT)

## QA status and methodology label

The full run is an engineering-complete approximation: graph parameters are fixed at $\alpha=\beta=0.1$. The paper's 10% validation score is not disclosed, and MOSEK is unavailable.

In [ ]:
qa = json.loads((OUTPUTS / 'qa' / 'full_fixed_qa.json').read_text())
display(pd.Series({
    'status': qa['status'],
    'milestone': qa['milestone'],
    'alpha': qa['alpha'],
    'beta': qa['beta'],
    'methodology_deviation': qa['methodology_deviation'],
}).to_frame('value'))
display(pd.DataFrame(qa['blocks']))
display(pd.Series(qa['later_graph_diagnostics']).to_frame('value'))

## Full OOS metrics

In [ ]:
metrics = pd.read_csv(OUTPUTS / 'results' / 'full_fixed_metrics.csv')
summary_columns = ['panel', 'strategy', 'annualized_mean_return', 'annualized_volatility', 'sharpe', 'maximum_drawdown', 'hit_rate']
display(metrics[summary_columns].round(4))

In [ ]:
raw = pd.read_parquet(OUTPUTS / 'results' / 'full_fixed_raw_returns.parquet')
scaled = pd.read_parquet(OUTPUTS / 'results' / 'full_fixed_vol_scaled_returns.parquet')
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
(1 + raw.fillna(0)).cumprod().plot(ax=axes[0], title='Raw strategy cumulative wealth')
(1 + scaled.fillna(0)).cumprod().plot(ax=axes[1], title='Portfolio-level 15% volatility-scaled cumulative wealth')
for ax in axes:
    ax.set_ylabel('growth of 1')
    ax.grid(alpha=0.25)
plt.tight_layout();

## Expanding-window coefficient paths

In [ ]:
coefficients = pd.read_csv(OUTPUTS / 'results' / 'full_fixed_coefficients.csv')
gmom_coefficients = coefficients[coefficients['model'].eq('GMOM')].set_index('block')
display(gmom_coefficients.round(4))
gmom_coefficients.drop(columns=['model', 'intercept']).plot(kind='bar', figsize=(13, 5), title='GMOM pooled OLS coefficients by test block')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout();

## Paper comparison

In [ ]:
gmom_raw = metrics[(metrics['panel'] == 'raw') & (metrics['strategy'] == 'GMOM')].iloc[0]
gmom_scaled = metrics[(metrics['panel'] == 'portfolio_vol_scaled') & (metrics['strategy'] == 'GMOM')].iloc[0]
comparison = pd.DataFrame({
    'current_fixed_param_run': [gmom_raw['annualized_mean_return'], gmom_raw['sharpe'], gmom_scaled['annualized_mean_return'], gmom_scaled['annualized_volatility'], gmom_scaled['sharpe']],
    'paper': [0.070, 1.363, 0.222, 0.147, 1.511],
}, index=['raw_return', 'raw_sharpe', 'scaled_return', 'scaled_volatility', 'scaled_sharpe'])
display(comparison.round(4))

## Fidelity checkpoint

The full causal chain is operational and all five walk-forward blocks pass. A strict paper label still requires the authors' graph-validation scoring rule and preferably MOSEK. Do not tune parameters against the 2000–2022 OOS metrics shown above.